# PyTorch: tamaño de capa oculta

Experimento de la bitácora DL04 asignada a Keber. Compara pérdida de entrenamiento y duración para seis tamaños de capa oculta; no calcula una métrica de prueba comparable. Requiere trainloader y las dependencias de la bitácora.


## Arquitectura y entrenamiento


In [ ]:
import torch
from torch import nn, optim
import torch.nn.functional as F
from torchvision import datasets, transforms
import time

transform = transforms.Compose([transforms.ToTensor(),
                              transforms.Normalize((0.5,), (0.5,)),
                              ])
# Bajamos y cargamos los datos.
trainset = datasets.MNIST('~/.pytorch/MNIST_data/', download=True, train=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)

In [ ]:
def create_model(hidden_units):
    return nn.Sequential(
        nn.Linear(784, hidden_units),
        nn.ReLU(),
        nn.Linear(hidden_units, 64),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(64, 10),
        nn.Dropout(0.2),
        nn.LogSoftmax(dim=1)
    )

In [ ]:
# parámetros
hidden_sizes = [16, 32, 64, 128, 256, 512]
epochs = 50
criterion = nn.NLLLoss()


In [ ]:
results = {}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if (torch.cuda.is_available()): print("utilizando GPU")

for h in hidden_sizes:
    print(f'Entrenando modelo con {h} unidades ocultas...')
    model = create_model(h).to(device)
    optimizer = optim.SGD(model.parameters(), lr=0.01)

    train_losses = []

    start_time = time.time()

    for e in range(epochs):
        running_loss = 0
        for images, labels in trainloader:
            images, labels = images.to(device), labels.to(device)
            images = images.view(images.shape[0], -1)

            output = model(images)
            loss = criterion(output, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        train_losses.append(running_loss/len(trainloader))
        print(f"Training loss: {running_loss/len(trainloader)}")
        running_loss = 0

    duration = time.time() - start_time
    print(f'Tiempo total para {h} unidades ocultas y {e} epochs: {duration:.2f} segundos')

    results[h] = {'losses': train_losses, 'duration': duration}


## Visualización de pérdida y tiempo


In [ ]:
import matplotlib.pyplot as plt

for h in hidden_sizes:
    plt.plot(results[h]['losses'], label=f'{h} hidden units')
plt.xlabel("Epochs")
plt.ylabel("Training Loss")
plt.title("Pérdida de entrenamiento por tamaño de capa oculta")
plt.legend()
plt.show()

In [ ]:
duraciones = [results[h]['duration'] for h in hidden_sizes]

plt.bar([str(h) for h in hidden_sizes], duraciones)
plt.xlabel("Unidades en capa oculta")
plt.ylabel("Tiempo (segundos)")
plt.title("Duración del entrenamiento por arquitectura")
plt.show()

In [ ]:
import pandas as pd

# Crear una lista con las filas de la tabla
resumen = []

for h in results:
    ultima_perdida = results[h]['losses'][-1]
    duracion = results[h]['duration']
    resumen.append({'Hidden Units': h,
                    'Última pérdida': round(ultima_perdida, 4),
                    'Tiempo total (s)': round(duracion, 2)})

# Crear DataFrame ordenado por tamaño de capa
df_resumen = pd.DataFrame(resumen).sort_values(by='Hidden Units')

# Mostrar
print(df_resumen.to_string(index=False))